# ML Zoomcamp 2026 — HW1

Notebook ini hanya mengerjakan Module 1, Q1–Q7. Setiap jawaban dihitung dari dataset resmi 2026; output disimpan setelah seluruh cell dijalankan dari awal.

- [Instruksi resmi](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/01-intro/homework.md)
- [Form submission](https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw01)
- [Dataset resmi](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv)

Jalankan cell berurutan. Dataset dibaca dari file di sebelah notebook sehingga perhitungan tidak bergantung pada koneksi internet.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import numpy as np
import pandas as pd

dataset_path = Path("car_fuel_efficiency_2026.csv")
dataset_sha256 = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
df = pd.read_csv(dataset_path)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Dataset SHA-256:", dataset_sha256)
print("Shape:", df.shape)
print(df.head().to_string(index=False))

Python: 3.14.5
NumPy: 2.5.3
Dataset SHA-256: 00a5cab178a8b7cd6e9157ee71dabc236ba7f20b1832336d358b07af14ba431f
Shape: (10000, 11)
 model_year origin fuel_type        drivetrain  num_doors  engine_displacement  num_cylinders  horsepower  vehicle_weight  acceleration  fuel_efficiency_mpg
       2006 Europe  Gasoline Front-wheel drive          4                 2180              6       243.0            3870           NaN                 31.9
       2008 Europe    Diesel Front-wheel drive          4                 2390              6       272.0            4210           NaN                 31.3
       1996   Asia  Gasoline Front-wheel drive          5                 2320              6       267.0            4240          17.3                 27.5
       1989 Europe  Gasoline Front-wheel drive          4                 2130              6       258.0            4490          18.7                 28.5
       1994    USA    Diesel Front-wheel drive          3                 2580        

## Q1 — Versi Pandas

Yang diminta adalah versi Pandas pada environment yang menjalankan notebook ini. Versi ini bisa berbeda pada komputer lain.

In [2]:
q1 = pd.__version__
print("Q1:", q1)

Q1: 3.0.5


## Q2 — Jumlah record

Satu baris data adalah satu record; header CSV tidak dihitung. `shape[0]` mengambil jumlah baris, bukan jumlah kolom.

In [3]:
q2 = int(df.shape[0])
print("Q2:", q2)

Q2: 10000


## Q3 — Jumlah jenis bahan bakar

Hitung kategori berbeda di `fuel_type`, bukan banyaknya mobil pada setiap kategori. Daftar dan frekuensi kategori ditampilkan sebagai bukti.

In [4]:
q3 = int(df["fuel_type"].nunique())
print(df["fuel_type"].value_counts().to_string())
print("Q3:", q3)

fuel_type
Gasoline    6913
Diesel      2053
Hybrid      1034
Q3: 3


## Q4 — Kolom yang mempunyai nilai kosong

`isna().sum()` menghitung nilai kosong per kolom. Soal meminta berapa kolom yang mempunyai setidaknya satu nilai kosong, bukan jumlah seluruh nilai kosong.

In [5]:
missing_by_column = df.isna().sum()
q4 = int((missing_by_column > 0).sum())
print(missing_by_column.to_string())
print("Q4:", q4)

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
Q4: 2


## Q5 — Efisiensi maksimum untuk mobil Asia

Filter `origin == "Asia"`, lalu ambil maksimum `fuel_efficiency_mpg`. Filter mempertahankan urutan baris asli; subset ini juga digunakan pada Q7.

In [6]:
asia = df.loc[df["origin"] == "Asia"]
assert not asia.empty
q5 = float(asia["fuel_efficiency_mpg"].max())
print("Jumlah mobil Asia:", len(asia))
print("Q5:", q5)

Jumlah mobil Asia: 2994
Q5: 41.2


## Q6 — Median horsepower sebelum dan sesudah imputasi

Median adalah nilai tengah; mode adalah nilai paling sering muncul. Isi nilai kosong dengan mode, lalu bandingkan median. Gunakan Series baru agar dataset asli tetap tersedia untuk pemeriksaan.

In [7]:
horsepower = df["horsepower"]
median_before = float(horsepower.median())
modes = horsepower.mode()
assert len(modes) == 1, "Periksa aturan pemilihan bila mode tidak unik."
horsepower_mode = float(modes.iloc[0])
filled_horsepower = horsepower.fillna(horsepower_mode)
assert filled_horsepower.isna().sum() == 0
median_after = float(filled_horsepower.median())

if median_after > median_before:
    q6 = "Yes, it increased"
elif median_after < median_before:
    q6 = "Yes, it decreased"
else:
    q6 = "No"

print("Nilai kosong sebelum:", int(horsepower.isna().sum()))
print("Median sebelum:", median_before)
print("Mode:", horsepower_mode)
print("Median sesudah:", median_after)
print("Q6:", q6)

Nilai kosong sebelum: 877
Median sebelum: 254.0
Mode: 252.0
Median sesudah: 252.0
Q6: Yes, it decreased


## Q7 — Jumlah bobot regresi

Ambil tujuh baris Asia pertama pada urutan dataset asli. Susun fitur dalam urutan `vehicle_weight`, lalu `model_year`. Dengan target `y` dari soal, hitung `w = (X.T @ X)^(-1) @ X.T @ y` tanpa menambah intercept atau melakukan scaling.

`X` berukuran 7 × 2, `XTX` berukuran 2 × 2, dan `w` mempunyai dua elemen. Hasil metode soal dibandingkan dengan `lstsq` sebagai pemeriksaan numerik independen.

In [8]:
first_seven_asia = asia[["vehicle_weight", "model_year"]].head(7)
X = first_seven_asia.to_numpy()
y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])
assert X.shape == (7, 2)
assert np.isfinite(X).all()
assert np.linalg.matrix_rank(X) == 2

XTX = X.T @ X
XTX_inv = np.linalg.inv(XTX)
w = XTX_inv @ X.T @ y
weight_sum = float(w.sum())
w_check = np.linalg.lstsq(X, y, rcond=None)[0]
np.testing.assert_allclose(w, w_check, rtol=1e-8, atol=1e-8)

choices_q7 = [0.0369, 0.369, 3.69, 36.9]
q7 = min(choices_q7, key=lambda option: abs(option - weight_sum))
assert abs(q7 - weight_sum) < 0.001

print("Index asli:", first_seven_asia.index.tolist())
print("X:\n", X)
print("y:", y)
print("XTX:\n", XTX)
print("Inverse XTX:\n", XTX_inv)
print("w:", w)
print("w dari lstsq:", w_check)
print("Jumlah bobot (presisi penuh):", repr(weight_sum))
print("Q7 (pilihan submission):", q7)

Index asli: [2, 6, 8, 10, 15, 16, 22]
X:
 [[4240 1996]
 [4310 1992]
 [4230 1997]
 [4420 1978]
 [3950 2007]
 [4750 2021]
 [4450 2019]]
y: [1100 1300  800  900 1000 1100 1200]
XTX:
 [[131950500  60750580]
 [ 60750580  28041424]]
Inverse XTX:
 [[ 2.96830537e-06 -6.43071025e-06]
 [-6.43071025e-06  1.39675281e-05]]
w: [0.13644777 0.2327492 ]
w dari lstsq: [0.13644777 0.2327492 ]
Jumlah bobot (presisi penuh): 0.36919696904925486
Q7 (pilihan submission): 0.369


## Jawaban submission dan bukti eksekusi

Nilai pada tabel berikut berasal dari variabel hasil perhitungan, bukan jawaban yang diketik manual. Q7 memakai pilihan resmi terdekat terhadap nilai berpresisi penuh. Cell ini menyimpan jawaban, versi environment, hash dataset, serta hasil antara ke `answers.json`.

In [9]:
answers = {"Q1": q1, "Q2": q2, "Q3": q3, "Q4": q4,
           "Q5": q5, "Q6": q6, "Q7": q7}
print(pd.Series(answers, name="Jawaban submission").to_string())

evidence = {
    "course": "ML Zoomcamp 2026",
    "homework": "HW1",
    "dataset_url": "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv",
    "dataset_sha256": dataset_sha256,
    "environment": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
    "answers": answers,
    "missing_by_column": {key: int(value) for key, value in missing_by_column.items()},
    "horsepower": {"median_before": median_before, "mode": horsepower_mode, "median_after": median_after},
    "regression": {"row_indices": first_seven_asia.index.tolist(), "X": X.tolist(),
                   "y": y.tolist(), "XTX": XTX.tolist(), "w": w.tolist(),
                   "weight_sum": weight_sum, "lstsq_w": w_check.tolist()}
}
Path("answers.json").write_text(json.dumps(evidence, indent=2) + "\n", encoding="utf-8")
print("Bukti disimpan: answers.json")

Q1                3.0.5
Q2                10000
Q3                    3
Q4                    2
Q5                 41.2
Q6    Yes, it decreased
Q7                0.369
Bukti disimpan: answers.json
